```
# Lab type:  debug
# Course:    NL301 Natural Language Processing with Python
# Lesson:    05 — Text Classification Pipelines
# Task:      Find and fix three bugs in a spam classifier (imbalanced dataset).
```

## Setup and reference code

In [ ]:
!pip install scikit-learn numpy --quiet
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, f1_score,
                             precision_recall_curve)

np.random.seed(42)

# Imbalanced dataset: 950 legitimate + 50 spam
legit = [f"meeting at {i} am please confirm agenda item {i}" for i in range(950)]
spam  = [f"winner free prize claim now click here {i}" for i in range(50)]
texts  = legit + spam
labels = [0]*950 + [1]*50

# Reference — correct pipeline with cross-validation and F1 scoring
pipe_ref = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=1000)),
    ('clf',   LogisticRegression(class_weight='balanced')),
])
scores = cross_val_score(pipe_ref, texts, labels, cv=5, scoring='f1')
print(f"Reference CV F1: {scores.mean():.3f} ± {scores.std():.3f}")


---
## Bug 1: Leakage — fit_transform before train_test_split

The vectorizer is fitted on all data before splitting. Test documents influence the vocabulary and IDF weights, making accuracy artificially high.

In [ ]:
# BUG: vectorizer fitted on full dataset before split
vectorizer_bug = TfidfVectorizer(max_features=1000)
X_all = vectorizer_bug.fit_transform(texts)              # ← Bug: leakage

X_train, X_test, y_train, y_test = train_test_split(
    X_all, labels, test_size=0.2, random_state=42)

clf_bug = LogisticRegression()
clf_bug.fit(X_train, y_train)
preds = clf_bug.predict(X_test)
print("Buggy accuracy:", clf_bug.score(X_test, y_test))
print("Buggy F1:      ", f1_score(y_test, preds))


**Explain the bug** — why does fitting the vectorizer on all data before splitting leak information? How does it inflate the reported F1?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Bug 1</summary>

**What the bug does:** `TfidfVectorizer.fit_transform(texts)` is called on all 1,000 documents before `train_test_split`. The fitted vocabulary and IDF weights incorporate statistics from test documents. At evaluation time, test text representations reflect their own presence in the IDF calculation, producing inflated scores that won't hold on truly unseen data.

**Correct approach:** Use a sklearn `Pipeline` passed to `cross_val_score`. Inside each fold, `fit_transform` is called only on the training portion of that fold; test documents go through `transform` using the fold's training-fitted vocabulary. This is the only way to guarantee a clean evaluation boundary.

</details>

In [ ]:
# FIX 1: Pipeline + cross_val_score so fitting only ever sees training folds
pipe_fix1 = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=1000)),
    ('clf',   LogisticRegression()),
])
scores = cross_val_score(pipe_fix1, texts, labels, cv=5, scoring='f1')
print(f"Fixed CV F1: {scores.mean():.3f} ± {scores.std():.3f}")


---
## Bug 2: Accuracy on imbalanced data

With 950 legitimate and 50 spam emails, a classifier that always predicts 'legitimate' gets 95% accuracy while completely failing to detect spam.

In [ ]:
# BUG: accuracy as the sole metric on imbalanced data
pipe_bug2 = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=1000)),
    ('clf',   LogisticRegression()),
])
X_tr, X_te, y_tr, y_te = train_test_split(texts, labels, test_size=0.2, random_state=42)
pipe_bug2.fit(X_tr, y_tr)
print("Buggy accuracy:", pipe_bug2.score(X_te, y_te))    # ← misleading

# Majority-class baseline
majority_preds = [0] * len(y_te)
print("Majority-class accuracy:", sum(1 for a, b in zip(majority_preds, y_te) if a == b) / len(y_te))
print("Majority-class F1 (spam):", f1_score(y_te, majority_preds, zero_division=0))


**Explain the bug** — why is accuracy misleading here? What does the majority-class baseline reveal?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Bug 2</summary>

**What the bug does:** With 950 legitimate and 50 spam emails, predicting "legitimate" for every sample achieves 95% accuracy while catching zero spam. The majority-class baseline confirms this: it scores 0.95 accuracy and 0.0 F1 for the spam class. Reporting accuracy alone makes the model appear to work when it entirely fails on the class that matters.

**Correct approach:** Evaluate with macro F1 or the minority-class F1 directly. Add `class_weight='balanced'` to `LogisticRegression` so the classifier penalises spam misclassifications proportionally to their rarity, rather than optimising for the dominant class.

</details>

In [ ]:
# FIX 2: use F1 and class_weight='balanced'
pipe_fix2 = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=1000)),
    ('clf',   LogisticRegression(class_weight='balanced')),
])
pipe_fix2.fit(X_tr, y_tr)
preds_fix2 = pipe_fix2.predict(X_te)
print("Fixed F1 (spam):", f1_score(y_te, preds_fix2))
print(classification_report(y_te, preds_fix2, target_names=['legit', 'spam']))


---
## Bug 3: Default threshold 0.5 misses minority class

For imbalanced classifiers, `predict()` uses threshold 0.5. When the classifier is uncertain, spam probabilities cluster below 0.5 — use `precision_recall_curve` to find the optimal threshold.

In [ ]:
# BUG: default 0.5 threshold on an imbalanced classifier
pipe_bug3 = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=1000)),
    ('clf',   LogisticRegression()),
])
pipe_bug3.fit(X_tr, y_tr)
proba = pipe_bug3.predict_proba(X_te)[:, 1]

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.hist(proba[np.array(y_te) == 1], bins=20, alpha=0.7, label='spam')
plt.hist(proba[np.array(y_te) == 0], bins=20, alpha=0.7, label='legit')
plt.axvline(0.5, color='red', linestyle='--', label='default threshold')
plt.xlabel('Predicted spam probability')
plt.legend()
plt.title('Probability distribution — default threshold may miss spam')
plt.tight_layout()
plt.savefig('/tmp/threshold_dist.png', dpi=80)
print("Plot saved to /tmp/threshold_dist.png")
print("F1 at default 0.5:", f1_score(y_te, (proba >= 0.5).astype(int)))


**Explain the bug** — what happens when spam probabilities cluster below 0.5? How does threshold selection affect precision vs recall?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Bug 3</summary>

**What the bug does:** On an imbalanced dataset with no `class_weight` adjustment, the classifier learns that spam is rare and assigns spam probabilities that cluster well below 0.5. Hard-predicting at 0.5 therefore misclassifies most spam as legitimate. The default threshold is a poor choice whenever the class distributions are unequal or the cost of false negatives differs from false positives.

**Correct approach:** Use `precision_recall_curve` to sweep all possible thresholds and find the one that maximises F1 (or the precision/recall trade-off your use case demands). In a spam filter you typically lower the threshold to increase recall — catching more spam — accepting some false positives. The optimal threshold is a business decision; `precision_recall_curve` gives you the data to make it.

</details>

In [ ]:
# FIX 3: use precision_recall_curve to find the optimal threshold
precisions, recalls, thresholds = precision_recall_curve(y_te, proba)
f1_scores = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-9)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
print(f"Default threshold 0.5  → F1: {f1_score(y_te, (proba >= 0.50).astype(int)):.3f}")
print(f"Optimal threshold {best_threshold:.3f} → F1: {f1_scores[best_idx]:.3f}")
print(f"  Precision: {precisions[best_idx]:.3f}  Recall: {recalls[best_idx]:.3f}")


---
## Summary

1. Bug 1 fix: ___
2. Bug 2 fix: ___
3. Bug 3 fix: ___

<details>
<summary>🔑 Reveal summary answers</summary>

1. **Bug 1:** Use a sklearn `Pipeline` inside `cross_val_score` so `fit_transform` never runs on test data.
2. **Bug 2:** Replace accuracy with macro F1; add `class_weight='balanced'` to surface and penalise minority-class failures.
3. **Bug 3:** Use `precision_recall_curve` to find the F1-optimal threshold rather than relying on the 0.5 default.

</details>